# Full Pipeline Walkthrough — `ffhq_00809_face`

Complete pipeline from raw FFHQ to paper figures. **Every cell produces a figure.**

## §0 Setup

In [ ]:
import os, sys, json, warnings
warnings.filterwarnings('ignore')
import numpy as np
import cv2; cv2.setNumThreads(0)
import matplotlib; matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Rectangle
from PIL import Image
from scipy.ndimage import sobel
import torch, torch.nn as nn, torch.nn.functional as F

ROOT = r'D:\zmm\facelift_pipeline'
DATA = os.path.join(ROOT, 'data', 'dataset')
CKPT = os.path.join(ROOT, 'checkpoints')
if not os.path.isdir(CKPT):
    CKPT = os.path.join(os.path.dirname(ROOT), 'checkpoints')
OUT  = os.path.join(ROOT, 'eval', 'paper_figures_final')
os.makedirs(OUT, exist_ok=True)

SAMPLE = 'ffhq_00809_face'
SPLIT  = 'val'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'cm',
    'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10,
    'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 8.5,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.03,
    'axes.spines.top': False, 'axes.spines.right': False,
})

# Low-saturation CVPR-style palette
C_OURS = '#4472C4'; C_OURS2 = '#7A9FD4'   # muted blue
C_PIX  = '#C55A11'; C_PIX2  = '#D4956A'   # muted orange/tan
C_SGN  = '#BFA84F'; C_BIC   = '#888888'; C_EXT = '#8B6A8B'  # muted gold/gray/mauve
DEPTH_CMAP = 'inferno'

def load_depth(path):
    img = np.asarray(Image.open(path)).astype(np.float32)
    return img / 65535.0 if img.max() > 255 else img / 255.0

def load_rgb(path):
    return np.array(Image.open(path).convert('RGB'))

def depth_range(d, bg=0.3):
    fg = d[d > bg]
    if len(fg) == 0: fg = d[d > 0.05]
    return (float(np.percentile(fg, 1)), float(np.percentile(fg, 99))) if len(fg) > 0 else (0, 1)

def to_depth_rgb(d, vmin, vmax):
    norm = np.clip((d - vmin) / (vmax - vmin + 1e-8), 0, 1)
    return plt.cm.inferno(norm)[..., :3]

def sobel_mag(d):
    return np.sqrt(sobel(d, axis=1)**2 + sobel(d, axis=0)**2)

def shaded_surface(d, light=(0.3, 0.3, 1.0), scale=200.0):
    dy, dx = np.gradient(d, axis=0), np.gradient(d, axis=1)
    nx, ny, nz = -dx*scale, -dy*scale, np.ones_like(d)
    norm = np.sqrt(nx**2 + ny**2 + nz**2) + 1e-8
    lx, ly, lz = [v / np.sqrt(sum(u**2 for u in light)) for v in light]
    return np.clip((nx/norm)*lx + (ny/norm)*ly + (nz/norm)*lz, 0, 1)

def save_fig(fig, name):
    for ext in ['pdf', 'png']:
        fig.savefig(os.path.join(OUT, f'{name}.{ext}'), dpi=300)
    print(f'  saved {name}')

print(f'Device: {device}, Sample: {SAMPLE}')

## Pipeline Flowchart
Full data flow: Raw FFHQ → FaceLift → Post-processing → Training → Evaluation

In [ ]:
# ═══ Pipeline figure — CVPR academic style ═══
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(14, 5.0))
ax.set_xlim(-0.5, 14.5); ax.set_ylim(-1.0, 5.2)
ax.axis('off'); ax.set_aspect('equal')

# ── Style ──
LW    = 0.9
EC    = '#2c2c2c'
FS    = 8.5
FS_SM = 7.0
FS_LB = 9.5

TINT_A = '#EDF2F9'
TINT_B = '#EEF5ED'
TINT_C = '#F5F0E8'
BOX_FC = '#FFFFFF'

def rbox(x, y, w, h, text, bold=False, fill=BOX_FC, ec_=EC):
    ax.add_patch(FancyBboxPatch((x, y - h/2), w, h,
                 boxstyle='round,pad=0.06', facecolor=fill, edgecolor=ec_,
                 linewidth=LW, zorder=3))
    ax.text(x + w/2, y, text, ha='center', va='center', fontsize=FS,
            fontweight='bold' if bold else 'normal', color='#111111', zorder=4,
            linespacing=1.25)

def harrow(x1, y, x2, label=''):
    mid = (x1 + x2) / 2
    gap = x2 - x1
    ax.annotate('', xy=(x2, y), xytext=(x1, y),
                arrowprops=dict(arrowstyle='->', color='#3a3a3a', lw=0.9))
    if label:
        ax.text(mid, y + 0.18, label, ha='center', fontsize=FS_SM,
                color='#555555', style='italic')

def varrow(x, y1, y2):
    ax.annotate('', xy=(x, y2), xytext=(x, y1),
                arrowprops=dict(arrowstyle='->', color='#3a3a3a', lw=0.9))

def stage_bg(x, y, w, h, color):
    ax.add_patch(FancyBboxPatch((x, y), w, h,
                 boxstyle='round,pad=0.12', facecolor=color, edgecolor='none',
                 linewidth=0, zorder=0, alpha=0.55))

# ════════════════════════════════════════════════════
# Row A — Data Preparation  (y = 4.0)
# ════════════════════════════════════════════════════
ya = 4.0
stage_bg(-0.3, ya - 0.50, 14.4, 1.05, TINT_A)
ax.text(0.0, ya + 0.42, '(a) Data Preparation', fontsize=FS_LB,
        fontweight='bold', color='#222222', fontstyle='italic')

rbox(0.0, ya, 1.5, 0.58, 'Raw FFHQ\n70k images')
harrow(1.5, ya, 2.1, 'crop')
rbox(2.1, ya, 1.6, 0.58, 'Align + White\nBackground')
harrow(3.7, ya, 4.3)
rbox(4.3, ya, 1.5, 0.58, 'Face\n$512{\\times}512$')
harrow(5.8, ya, 6.4)
rbox(6.4, ya, 2.0, 0.58, 'FaceLift 3DGS', bold=True, fill='#E8EEF7')
harrow(8.4, ya, 9.0)
rbox(9.0, ya, 2.3, 0.58, 'Renders $1024^2$\nRGB / D / N / O')

# ════════════════════════════════════════════════════
# Row B — Output Cleaning  (y = 2.65)
# ════════════════════════════════════════════════════
yb = 2.65
stage_bg(-0.3, yb - 0.50, 14.4, 1.05, TINT_B)
ax.text(0.0, yb + 0.42, '(b) Output Cleaning', fontsize=FS_LB,
        fontweight='bold', color='#222222', fontstyle='italic')

rbox(0.0, yb, 1.5, 0.58, 'Raw Renders')
harrow(1.5, yb, 2.1, 'align')
rbox(2.1, yb, 1.7, 0.58, 'Postprocess\n(norm / fill)')
harrow(3.8, yb, 4.4)
rbox(4.4, yb, 1.4, 0.58, 'Clean Maps')
harrow(5.8, yb, 6.4, '90/10')
rbox(6.4, yb, 1.5, 0.58, 'Dataset\n1159 / 129')

# Fan-out: three outputs stacked to the right
fan_x = 8.6
rbox(fan_x, yb + 0.58, 1.5, 0.44, 'LR 256 (8-bit)')
rbox(fan_x, yb,        1.5, 0.44, 'Face Mask')
rbox(fan_x, yb - 0.58, 1.5, 0.44, 'DSINE Normal')
for dy in [0.58, 0, -0.58]:
    ax.annotate('', xy=(fan_x, yb + dy), xytext=(7.9, yb),
                arrowprops=dict(arrowstyle='->', color='#3a3a3a', lw=0.7))

# ════════════════════════════════════════════════════
# Row C — Training + Evaluation  (y = 1.1)
# ════════════════════════════════════════════════════
yc = 1.1
stage_bg(-0.3, yc - 0.95, 14.4, 1.50, TINT_C)
ax.text(0.0, yc + 0.42, '(c) Training + Evaluation', fontsize=FS_LB,
        fontweight='bold', color='#222222', fontstyle='italic')

rbox(0.0, yc, 1.6, 0.58, 'LR 8-bit\n+ HR 16-bit')
harrow(1.6, yc, 2.2)
rbox(2.2, yc, 1.8, 0.58, 'Depth SR\nModel', bold=True, fill='#E8EEF7')
harrow(4.0, yc, 4.6)
rbox(4.6, yc, 1.7, 0.58, 'SR Output\n$1024^2$ 16-bit')

# 2D eval
harrow(6.3, yc, 6.9)
rbox(6.9, yc, 1.7, 0.58, 'Zone Eval\nPSNR / SSIM')

# 3D eval (below SR Output)
varrow(5.45, yc - 0.29, yc - 0.78)
rbox(4.6, yc - 1.08, 1.7, 0.46, '3D Mesh\nReconstruct')
harrow(6.3, yc - 1.08, 6.9)
rbox(6.9, yc - 1.08, 1.7, 0.46, 'F-score /\nHausdorff', bold=True, fill='#FDF3E7')

# Key finding callout — right side, clear of arrows
ax.text(9.3, yc - 0.25,
        'Key finding:\nhigh PSNR $\\neq$ good geometry\n(35 pp F-score gap)',
        fontsize=7.5, color='#333333', linespacing=1.4,
        bbox=dict(boxstyle='round,pad=0.25', fc='#F7F7F7', ec='#AAAAAA',
                  lw=0.7, alpha=0.9))

# ── Vertical: Row A → Row B ──
varrow(10.15, ya - 0.29, yb + 0.29)

# ── Vertical: Row B fan-out → Row C input (right-side path, no crossing) ──
# Small downward arrow on the left margin: conceptual link
ax.annotate('', xy=(0.8, yc + 0.29), xytext=(0.8, yb - 0.29),
            arrowprops=dict(arrowstyle='->', color='#888888', lw=0.7,
                            linestyle='--'))
ax.text(0.95, (yc + yb) / 2, 'feeds', fontsize=6, color='#888888',
        style='italic', va='center')

save_fig(fig, 'fig00_pipeline_flowchart')
plt.show()


## §1 Data Cleaning Step 1: Raw FFHQ → White Background Crop

**Input**: Raw FFHQ images (`data/raw_faces/`), varying sizes and backgrounds  
**Steps**:
1. **Face detection** — dlib/MediaPipe 68-landmark detection
2. **Alignment** — Affine alignment based on eye positions
3. **Crop** — Center on nose, crop to 512×512
4. **White background** — Fill non-face region with white (required by FaceLift)

**Output**: `data/cropped_faces/{name}_face.png` (512×512, RGB, white background)

**Why white background**: FaceLift's 3DGS reconstruction assumes a pure white background. Non-white regions are treated as geometry, causing splat scattering.

In [ ]:
raw_face = load_rgb(os.path.join(ROOT, 'data', 'raw_faces', 'ffhq_00809.png'))
cropped_face = load_rgb(os.path.join(ROOT, 'data', 'cropped_faces', f'{SAMPLE}.png'))

# Diff: where white bg was added
diff_mask = (cropped_face.mean(axis=-1) > 250).astype(float)  # white pixels

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
axes[0].imshow(raw_face)
axes[0].set_title(f'Original FFHQ\n{raw_face.shape[1]}x{raw_face.shape[0]}', fontsize=9)
axes[1].imshow(cropped_face)
axes[1].set_title(f'Cropped + white BG\n{cropped_face.shape[1]}x{cropped_face.shape[0]}', fontsize=9)
axes[2].imshow(diff_mask, cmap='gray', vmin=0, vmax=1)
axes[2].set_title('White BG region', fontsize=9)
for ax in axes: ax.axis('off')
fig.suptitle('§1  FFHQ original → cropped face with white background', fontsize=12, fontweight='bold')
fig.tight_layout()
save_fig(fig, 'fig01_original_vs_whitebg')
plt.show()

## §2 FaceLift 3DGS Rendering

**Input**: White-background face images (`data/cropped_faces/`)  
**FaceLift pipeline**:
1. Image → Multi-view diffusion (MVDiffusion) generates 6 views
2. 6 views → GS-LRM reconstructs 3D Gaussian Splatting
3. Render frontal view → RGB + depth (16-bit) + normal + opacity

**Output** (1024×1024):
- `data/rgb_rendered/` — Rendered RGB
- `data/depth_maps/` — Depth maps (I;16 PNG, per-view min/max normalized)
- `data/normal_maps/` — Normal maps (with per-splat aliasing)
- `data/opacity_maps/` — Opacity maps

**Command**: `facelift_pipeline.ipynb` Cell 16, `render_all_maps_gpu()`

In [ ]:
raw_rgb    = load_rgb(os.path.join(ROOT, 'data', 'rgb_rendered', f'{SAMPLE}.png'))
raw_depth  = load_depth(os.path.join(ROOT, 'data', 'depth_maps', f'{SAMPLE}.png'))
raw_normal = load_rgb(os.path.join(ROOT, 'data', 'normal_maps', f'{SAMPLE}.png'))
raw_opac   = load_depth(os.path.join(ROOT, 'data', 'opacity_maps', f'{SAMPLE}.png'))

vmin_r, vmax_r = depth_range(raw_depth)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
axes[0].imshow(raw_rgb); axes[0].set_title('RGB render')
axes[1].imshow(raw_depth, cmap=DEPTH_CMAP, vmin=vmin_r, vmax=vmax_r); axes[1].set_title('Depth (16-bit)')
axes[2].imshow(raw_normal); axes[2].set_title('Normal (3DGS)')
axes[3].imshow(raw_opac, cmap='gray'); axes[3].set_title('Opacity')
for ax in axes: ax.axis('off')
fig.suptitle('§2  FaceLift 3DGS raw rendering', fontsize=12, fontweight='bold')
fig.tight_layout()
save_fig(fig, 'fig02_3dgs_renders')
plt.show()

## §3 FaceLift Render vs Original Face

3DGS reconstruction quality check: rendered image vs original SSIM / difference map.  
**Note**: FaceLift output is 1024×1024 (original 512 → 2x upsample); fine details are diffusion-generated, not real.

In [ ]:
from skimage.metrics import structural_similarity as ssim

# Resize cropped to match render size
cropped_resized = np.array(Image.fromarray(cropped_face).resize(
    (raw_rgb.shape[1], raw_rgb.shape[0]), Image.BICUBIC))

diff_rgb = np.abs(cropped_resized.astype(float) - raw_rgb.astype(float))
diff_gray = diff_rgb.mean(axis=-1)
ssim_val, ssim_map = ssim(cropped_resized, raw_rgb, channel_axis=-1, full=True, data_range=255)

cy, cx, cs = 500, 512, 200
crop = (slice(cy-cs, cy+cs), slice(cx-cs, cx+cs))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
# Row 1: full
axes[0][0].imshow(cropped_resized); axes[0][0].set_title('Original (resized)')
axes[0][1].imshow(raw_rgb); axes[0][1].set_title('3DGS render')
axes[0][2].imshow(diff_gray, cmap='hot', vmin=0, vmax=60); axes[0][2].set_title('|Diff|')
axes[0][3].imshow(1 - ssim_map.mean(axis=-1), cmap='magma', vmin=0, vmax=0.3)
axes[0][3].set_title(f'1-SSIM (mean={ssim_val:.3f})')
# Row 2: crop
axes[1][0].imshow(cropped_resized[crop]); axes[1][0].set_title('Original (zoom)')
axes[1][1].imshow(raw_rgb[crop]); axes[1][1].set_title('Render (zoom)')
axes[1][2].imshow(diff_gray[crop], cmap='hot', vmin=0, vmax=60); axes[1][2].set_title('Diff (zoom)')
axes[1][3].imshow(1 - ssim_map[crop].mean(axis=-1), cmap='magma', vmin=0, vmax=0.3)
axes[1][3].set_title('1-SSIM (zoom)')
for row in axes:
    for ax in row: ax.axis('off')
axes[0][0].set_ylabel('Full', fontsize=10, rotation=90, labelpad=10)
axes[0][0].yaxis.set_visible(True); axes[0][0].tick_params(left=False, labelleft=False)
axes[1][0].set_ylabel('Crop', fontsize=10, rotation=90, labelpad=10)
axes[1][0].yaxis.set_visible(True); axes[1][0].tick_params(left=False, labelleft=False)
fig.suptitle(f'§3  Render vs original — SSIM={ssim_val:.3f}', fontsize=12, fontweight='bold')
fig.tight_layout()
save_fig(fig, 'fig03_render_vs_original')
plt.show()

## §4 3DGS Normal Aliasing\nPer-splat pits/bumps — confirmed by Normal-GS, 2DGS, SuGaR, DN-Splatter

In [ ]:
cy, cx, cs = 400, 512, 100
crop = (slice(cy-cs, cy+cs), slice(cx-cs, cx+cs))

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
axes[0].imshow(raw_rgb[crop]); axes[0].set_title('RGB (zoom)')
axes[1].imshow(raw_normal[crop]); axes[1].set_title('3DGS normal (zoom)')
n_gray = np.mean(raw_normal[crop].astype(float), axis=-1)
g = sobel_mag(n_gray / 255.0)
axes[2].imshow(g, cmap='hot', vmin=0, vmax=np.percentile(g, 98))
axes[2].set_title('Normal gradient\n(per-splat aliasing)')
for ax in axes: ax.axis('off')
fig.suptitle('§4  3DGS normal aliasing — unusable as supervision', fontsize=11, fontweight='bold')
fig.tight_layout()
save_fig(fig, 'fig04_normal_aliasing')
plt.show()

## §5 Data Cleaning Step 2: FaceLift Output Post-processing

**Input**: Raw 3DGS renders (`data/rgb_rendered/`, `data/depth_maps/`, ...)  
**Post-processing steps** (`scripts/postprocess_maps.py`):
1. **Landmark alignment** — Fix alignment offset between render and original crop via affine transform
2. **Nose-relative depth normalization** — Use nose tip as anchor, percentile 1–99.9 normalization to [0,1]
3. **Normal bilateral filtering** — Bilateral filter to smooth per-splat aliasing (partial fix only)
4. **Hole filling** — Small holes via morphological closing, large holes via distance-transform NN fill (max_hole_area=5000)
5. **Depth-normal consistency check** — Cosine similarity > 0.9 required

**Command**:
```
python scripts/run_postprocess.py
```

**Output**: `data/postprocessed/{rgb,depth,normal,opacity}/`

In [ ]:
post_rgb    = load_rgb(os.path.join(ROOT, 'data', 'postprocessed', 'rgb', f'{SAMPLE}.png'))
post_depth  = load_depth(os.path.join(ROOT, 'data', 'postprocessed', 'depth', f'{SAMPLE}.png'))
post_normal = load_rgb(os.path.join(ROOT, 'data', 'postprocessed', 'normal', f'{SAMPLE}.png'))
post_opac   = load_depth(os.path.join(ROOT, 'data', 'postprocessed', 'opacity', f'{SAMPLE}.png'))
vmin_p, vmax_p = depth_range(post_depth)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes[0][0].imshow(raw_rgb); axes[0][0].set_title('RGB (raw)')
axes[0][1].imshow(raw_depth, cmap=DEPTH_CMAP, vmin=vmin_r, vmax=vmax_r); axes[0][1].set_title('Depth (raw)')
axes[0][2].imshow(raw_normal); axes[0][2].set_title('Normal (raw)')
axes[0][3].imshow(raw_opac, cmap='gray'); axes[0][3].set_title('Opacity (raw)')
axes[1][0].imshow(post_rgb); axes[1][0].set_title('RGB (aligned)')
axes[1][1].imshow(post_depth, cmap=DEPTH_CMAP, vmin=vmin_p, vmax=vmax_p); axes[1][1].set_title('Depth (normalized)')
axes[1][2].imshow(post_normal); axes[1][2].set_title('Normal (smoothed)')
axes[1][3].imshow(post_opac, cmap='gray'); axes[1][3].set_title('Opacity')
for row in axes:
    for ax in row: ax.axis('off')
axes[0][0].set_ylabel('Raw', fontsize=10, rotation=90, labelpad=10)
axes[0][0].yaxis.set_visible(True); axes[0][0].tick_params(left=False, labelleft=False)
axes[1][0].set_ylabel('Post', fontsize=10, rotation=90, labelpad=10)
axes[1][0].yaxis.set_visible(True); axes[1][0].tick_params(left=False, labelleft=False)
fig.suptitle('§5  Raw vs Postprocessed', fontsize=12, fontweight='bold')
fig.tight_layout()
save_fig(fig, 'fig05_postprocessing')
plt.show()

## §6 Data Cleaning Step 3: Face Mask Generation

**Purpose**: SR evaluation only within the face region (zone-aware evaluation)

**Steps** (`scripts/make_face_masks.py`):
1. Extract foreground from `cropped_faces` — non-white pixels
2. Morphological operations — erode + dilate to clean edge artifacts
3. Generate binary mask — 0/255, 1024×1024

**Command**:
```
python scripts/make_face_masks.py
```

**Output**: `data/dataset/{split}/mask/`

In [ ]:
mask_hr = np.asarray(Image.open(os.path.join(DATA, SPLIT, 'mask', f'{SAMPLE}.png')))
opac = post_opac
opac_thresh = (opac > 0.5).astype(np.uint8) * 255

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
axes[0].imshow(cropped_face); axes[0].set_title('Cropped face')
axes[1].imshow(opac, cmap='gray', vmin=0, vmax=1); axes[1].set_title('Opacity map')
axes[2].imshow(opac_thresh, cmap='gray', vmin=0, vmax=255); axes[2].set_title('Opacity > 0.5')
axes[3].imshow(mask_hr, cmap='gray', vmin=0, vmax=255); axes[3].set_title('Final face mask\n(morphology)')
for ax in axes: ax.axis('off')
fig.suptitle('§6  Face mask generation pipeline', fontsize=11, fontweight='bold')
fig.tight_layout()
save_fig(fig, 'fig06_mask_generation')
plt.show()

## §7 Data Cleaning Step 4: Dataset Assembly + LR/HR Generation

**Steps**:
1. **Reorganize** (`scripts/reorganize_dataset.py`) — 90/10 train/val split, seed=42 → 1159 train, 129 val
2. **LR/HR generation** (`scripts/make_lr_hr_pairs.py`) — HR 1024 16-bit → LR 256 8-bit (INTER_AREA downsample + uint8 quantization)
3. Also generates `depth_lr_16bit/` (256 uint16, spatial-only downsampling, for ablation)

**Key observation**: 8-bit quantization introduces banding artifacts (visible in bottom row, col 2 vs col 3)

**Command**:
```
python scripts/reorganize_dataset.py
python scripts/make_lr_hr_pairs.py
```

**Output**: `data/dataset/{train,val}/{depth, depth_lr_8bit, depth_lr_16bit, image, mask, normal, opacity}/`

In [ ]:
hr = load_depth(os.path.join(DATA, SPLIT, 'depth', f'{SAMPLE}.png'))
lr_8 = load_depth(os.path.join(DATA, SPLIT, 'depth_lr_8bit', f'{SAMPLE}.png'))
lr_16 = load_depth(os.path.join(DATA, SPLIT, 'depth_lr_16bit', f'{SAMPLE}.png'))
rgb = load_rgb(os.path.join(DATA, SPLIT, 'image', f'{SAMPLE}.png'))
mask = (np.asarray(Image.open(os.path.join(DATA, SPLIT, 'mask', f'{SAMPLE}.png'))) > 127).astype(np.float32)

vmin, vmax = depth_range(hr)
bic_8 = cv2.resize(lr_8.astype(np.float32), (1024,1024), interpolation=cv2.INTER_CUBIC).clip(0,1)
bic_16 = cv2.resize(lr_16.astype(np.float32), (1024,1024), interpolation=cv2.INTER_CUBIC).clip(0,1)
lr_nn = cv2.resize(lr_8, (1024,1024), interpolation=cv2.INTER_NEAREST)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes[0][0].imshow(rgb); axes[0][0].set_title('RGB (1024)')
axes[0][1].imshow(hr, cmap=DEPTH_CMAP, vmin=vmin, vmax=vmax); axes[0][1].set_title('HR depth (1024, 16-bit)')
axes[0][2].imshow(lr_8, cmap=DEPTH_CMAP); axes[0][2].set_title('LR depth (256, 8-bit)')
axes[0][3].imshow(mask, cmap='gray'); axes[0][3].set_title('Face mask')

# Nose/mouth crop — shows banding on smooth cheek
cy, cx, cs = 550, 512, 100
crop = (slice(cy-cs, cy+cs), slice(cx-cs, cx+cs))
axes[1][0].imshow(to_depth_rgb(hr, vmin, vmax)[crop]); axes[1][0].set_title('HR (16-bit)')
axes[1][1].imshow(to_depth_rgb(bic_8, vmin, vmax)[crop]); axes[1][1].set_title('Bicubic 8-bit\n(banding)')
axes[1][2].imshow(to_depth_rgb(bic_16, vmin, vmax)[crop]); axes[1][2].set_title('Bicubic 16-bit\n(no banding)')
axes[1][3].imshow(np.abs(bic_8 - hr)[crop], cmap='magma', vmin=0, vmax=0.01); axes[1][3].set_title('|8bit bic - HR|')
for row in axes:
    for ax in row: ax.axis('off')
axes[0][0].set_ylabel('Full', fontsize=10, rotation=90, labelpad=10)
axes[0][0].yaxis.set_visible(True); axes[0][0].tick_params(left=False, labelleft=False)
axes[1][0].set_ylabel('Crop', fontsize=10, rotation=90, labelpad=10)
axes[1][0].yaxis.set_visible(True); axes[1][0].tick_params(left=False, labelleft=False)
fig.suptitle('§7  LR/HR pairs — 8-bit quantization banding', fontsize=12, fontweight='bold')
fig.tight_layout()
save_fig(fig, 'fig07_dataset_lr_hr')
plt.show()

## §8 Data Cleaning Step 5: DSINE Pseudo-GT Normal

**Problem**: 3DGS-rendered normals have per-splat aliasing (§4), unusable as normal-aware loss supervision  
**Solution**: Use DSINE (CVPR 2024) monocular normal estimator for pseudo ground-truth normals

**Command**:
```
python scripts/compute_dsine_normals.py
```
~3 min for 1288 images

**Output**: `data/dataset/{split}/normal_dsine/`  
**Bottom row**: Chin/cheek crop comparing 3DGS normals (left, aliased) vs DSINE (right, clean)

In [ ]:
n3dgs  = load_rgb(os.path.join(DATA, SPLIT, 'normal', f'{SAMPLE}.png'))
ndsine = load_rgb(os.path.join(DATA, SPLIT, 'normal_dsine', f'{SAMPLE}.png'))
diff_n = np.sqrt(((n3dgs.astype(float)/255 - ndsine.astype(float)/255)**2).sum(axis=-1))

# Chin/cheek crop — shows smoothness difference best
cy, cx, cs = 620, 512, 180
crop = (slice(cy-cs, cy+cs), slice(cx-cs, cx+cs))

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
axes[0].imshow(rgb[crop]); axes[0].set_title('RGB')
axes[1].imshow(n3dgs[crop]); axes[1].set_title('3DGS normal\n(per-splat aliasing)')
axes[2].imshow(ndsine[crop]); axes[2].set_title('DSINE pseudo-GT\n(clean)')
im = axes[3].imshow(diff_n[crop], cmap='inferno', vmin=0, vmax=0.15)
axes[3].set_title('|3DGS - DSINE|')
fig.colorbar(im, ax=axes[3], shrink=0.8)
for ax in axes: ax.axis('off')
fig.suptitle('§8  Normal: 3DGS aliased vs DSINE clean (chin/cheek crop)', fontsize=11, fontweight='bold')
fig.tight_layout()
save_fig(fig, 'fig08_normal_3dgs_vs_dsine')
plt.show()

## §9 1D Depth Profile — 8-bit Quantization Steps

In [ ]:
row_y = 500
x_px = np.arange(300, 700)

fig, (ax, axins) = plt.subplots(1, 2, figsize=(10, 3.2), gridspec_kw={'width_ratios': [3, 1.3]})

# Main plot
ax.plot(x_px, hr[row_y, 300:700], 'k-', linewidth=1.5, label='HR 16-bit (GT)')
ax.plot(x_px, bic_8[row_y, 300:700], '--', color='gray', linewidth=1, label='Bicubic from 8-bit')
ax.plot(x_px, lr_nn[row_y, 300:700], '-', color='#E91E63', linewidth=0.7, alpha=0.7, label='LR nearest (8-bit)')
ax.set_xlabel('Pixel x'); ax.set_ylabel('Depth value')
ax.legend(fontsize=7.5, loc='lower right')

# Zoom region highlight on main plot
ax.axvspan(450, 530, alpha=0.08, color='gray')

# Separate zoom panel (right)
zi = (x_px >= 450) & (x_px <= 530)
axins.plot(x_px[zi], hr[row_y, x_px[zi]], 'k-', linewidth=1.5, label='HR')
axins.plot(x_px[zi], bic_8[row_y, x_px[zi]], '--', color='gray', linewidth=1, label='Bic 8-bit')
axins.plot(x_px[zi], lr_nn[row_y, x_px[zi]], '-', color='#E91E63', linewidth=0.8, alpha=0.7, label='LR 8-bit')
axins.set_title('Zoom: quantization steps', fontsize=8)
axins.tick_params(labelsize=7)
axins.set_xlabel('Pixel x', fontsize=8)

fig.suptitle('§9  1D depth profile — 8-bit quantization banding', fontsize=11, fontweight='bold')
fig.tight_layout()
save_fig(fig, 'fig09_depth_profile')
plt.show()


## §10 Load All Models

In [ ]:
sys.path.insert(0, os.path.join(ROOT, 'scripts'))
from train_depth_upres import DepthUpResUNet

models = {}
for name, ckdir, pn in [
    ('UNet', 'depth_upres_8bit_ch32', False),
    ('UNet+DSINE', 'depth_upres_8bit_ch32_normal_w050_dsine', True),
]:
    p = os.path.join(CKPT, ckdir, 'best.pt')
    if os.path.exists(p):
        ck = torch.load(p, map_location=device, weights_only=False)
        m = DepthUpResUNet(base_ch=32, predict_normal=pn)
        m.load_state_dict(ck['model'], strict=False)
        models[name] = m.to(device).eval()
        print(f'  loaded {name} (ep {ck.get("epoch","?")})')

class ResBlock(nn.Module):
    def __init__(self, nf, rs=0.1):
        super().__init__()
        self.body = nn.Sequential(nn.Conv2d(nf,nf,3,padding=1), nn.ReLU(True), nn.Conv2d(nf,nf,3,padding=1))
        self.rs = rs
    def forward(self, x): return x + self.body(x) * self.rs
class EDSR_Light(nn.Module):
    def __init__(self, nf=64, nb=16):
        super().__init__()
        self.head = nn.Conv2d(1, nf, 3, padding=1)
        self.body = nn.Sequential(*[ResBlock(nf) for _ in range(nb)])
        self.body_tail = nn.Conv2d(nf, nf, 3, padding=1)
        self.up = nn.Sequential(nn.Conv2d(nf, nf*4, 3, padding=1), nn.PixelShuffle(2),
                                nn.Conv2d(nf, nf*4, 3, padding=1), nn.PixelShuffle(2))
        self.tail = nn.Conv2d(nf, 1, 3, padding=1)
    def forward(self, x):
        h = self.head(x); h = h + self.body_tail(self.body(h)); h = self.up(h)
        bic = F.interpolate(x, scale_factor=4, mode='bicubic', align_corners=False).clamp(0,1)
        return (bic + self.tail(h) * 0.5).clamp(0,1)
class ResBlockBN(nn.Module):
    def __init__(self, nf):
        super().__init__()
        self.body = nn.Sequential(nn.Conv2d(nf,nf,3,padding=1), nn.BatchNorm2d(nf), nn.PReLU(),
                                  nn.Conv2d(nf,nf,3,padding=1), nn.BatchNorm2d(nf))
    def forward(self, x): return x + self.body(x)
class SRResNetM(nn.Module):
    def __init__(self, nf=64, nb=16):
        super().__init__()
        self.head = nn.Sequential(nn.Conv2d(1, nf, 9, padding=4), nn.PReLU())
        self.body = nn.Sequential(*[ResBlockBN(nf) for _ in range(nb)])
        self.body_tail = nn.Sequential(nn.Conv2d(nf, nf, 3, padding=1), nn.BatchNorm2d(nf))
        self.up = nn.Sequential(nn.Conv2d(nf, nf*4, 3, padding=1), nn.PixelShuffle(2), nn.PReLU(),
                                nn.Conv2d(nf, nf*4, 3, padding=1), nn.PixelShuffle(2), nn.PReLU())
        self.tail = nn.Conv2d(nf, 1, 9, padding=4)
    def forward(self, x):
        h = self.head(x); h = h + self.body_tail(self.body(h)); h = self.up(h)
        bic = F.interpolate(x, scale_factor=4, mode='bicubic', align_corners=False).clamp(0,1)
        return (bic + self.tail(h) * 0.5).clamp(0,1)

for cls, name, ckdir in [(EDSR_Light, 'EDSR', 'baseline_edsr'),
                          (SRResNetM, 'SRResNet', 'baseline_srresnet')]:
    p = os.path.join(CKPT, ckdir, 'best.pt')
    if os.path.exists(p):
        ck = torch.load(p, map_location=device, weights_only=False)
        m = cls(); m.load_state_dict(ck['model'], strict=False)
        models[name] = m.to(device).eval()
        print(f'  loaded {name} (ep {ck.get("epoch","?")})')

# SGNet (RGB-guided, num_feats=24)
sgnet_dir = os.path.join(ROOT, 'external', 'SGNet')
if not os.path.isdir(sgnet_dir):
    sgnet_dir = os.path.join(os.path.dirname(ROOT), 'external', 'SGNet')
sgnet_ck = os.path.join(CKPT, 'baseline_sgnet', 'best.pt')
if os.path.exists(sgnet_ck) and os.path.isdir(sgnet_dir):
    sys.path.insert(0, sgnet_dir)
    from models.SGNet import SGNet as SGNetCls
    ck = torch.load(sgnet_ck, map_location=device, weights_only=False)
    nf = ck.get('args', {}).get('num_feats', 24) if isinstance(ck.get('args'), dict) else 24
    m = SGNetCls(num_feats=nf, kernel_size=3, scale=4)
    m.load_state_dict(ck['model'], strict=False)
    models['SGNet'] = m.to(device).eval()
    print(f'  loaded SGNet (num_feats={nf}, ep {ck.get("epoch","?")})')

print(f'\nAll models: {list(models.keys())}')

## §11 Inference + Per-sample Metrics

In [ ]:
preds = {}
for name in ['EDSR', 'SRResNet', 'UNet', 'UNet+DSINE']:
    if name in models:
        with torch.no_grad():
            x = torch.from_numpy(lr_8).float().unsqueeze(0).unsqueeze(0).to(device)
            out = models[name](x)
            if isinstance(out, tuple): out = out[0]
            preds[name] = out.squeeze().cpu().numpy().clip(0, 1)
        print(f'  {name}: done')

# SGNet: forward((image, depth))
if 'SGNet' in models:
    with torch.no_grad():
        lr_t = torch.from_numpy(lr_8).float().unsqueeze(0).unsqueeze(0).to(device)
        rgb_f = rgb.astype(np.float32) / 255.0
        rgb_t = torch.from_numpy(rgb_f).permute(2,0,1).unsqueeze(0).float().to(device)
        out, _ = models['SGNet']((rgb_t, lr_t))
        preds['SGNet'] = out.squeeze().cpu().numpy().clip(0, 1)
    print(f'  SGNet: done')

print(f'\n{"Method":15s}  {"Masked L1":>10s}  {"PSNR (dB)":>10s}')
print('-' * 40)
for name in ['Bicubic', 'EDSR', 'SRResNet', 'SGNet', 'UNet', 'UNet+DSINE']:
    pred = bic_8 if name == 'Bicubic' else preds.get(name)
    if pred is None: continue
    m_px = mask > 0.5
    l1 = np.abs(pred[m_px] - hr[m_px]).mean()
    mse = ((pred[m_px] - hr[m_px]) ** 2).mean()
    psnr = 10 * np.log10(1.0 / max(mse, 1e-12))
    print(f'{name:15s}  {l1:10.5f}  {psnr:10.2f}')

## §12 Per-method Error Analysis
2D depth/gradient images look identical across methods (error is only ~0.002) — this is the core finding of the paper.
Instead we show: (a) error distribution histogram, (b) F-score bars, (c) PSNR bars — the difference becomes obvious.

In [ ]:
# §12: Per-method error analysis (replaces depth/gradient which look identical)
# Key insight: 2D images look the same, but error DISTRIBUTION differs

all_names = ['Bicubic', 'EDSR', 'SRResNet', 'SGNet', 'UNet', 'UNet+DSINE']
all_preds = {'Bicubic': bic_8}
all_preds.update(preds)
all_colors = {'Bicubic': C_BIC, 'EDSR': C_PIX, 'SRResNet': C_PIX2,
              'SGNet': C_SGN, 'UNet': C_OURS2, 'UNet+DSINE': C_OURS}

m_px = mask > 0.5
threshold = 1e-3

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# (a) Error histogram — shows tail behavior
for name in all_names:
    if name not in all_preds: continue
    err = np.abs(all_preds[name][m_px] - hr[m_px])
    axes[0].hist(err, bins=200, range=(0, 0.015), alpha=0.6,
                 color=all_colors[name], label=name, histtype='step', linewidth=1.5)
axes[0].axvline(threshold, color='red', linestyle='--', linewidth=1, label=f'threshold={threshold}')
axes[0].set_xlabel('|Error|')
axes[0].set_ylabel('Pixel count')
axes[0].set_title('(a) Error distribution\n(face region)', fontsize=10)
axes[0].legend(fontsize=7, loc='upper right')
axes[0].set_xlim(0, 0.015)

# (b) F-score bars — directly shows geometry quality
f_scores = {}
for name in all_names:
    if name not in all_preds: continue
    err = np.abs(all_preds[name][m_px] - hr[m_px])
    f_scores[name] = (err < threshold).mean()

names_f = list(f_scores.keys())
vals_f = [f_scores[n] for n in names_f]
colors_f = [all_colors[n] for n in names_f]
bars = axes[1].bar(names_f, vals_f, color=colors_f, edgecolor='black', linewidth=0.5)
for b, v in zip(bars, vals_f):
    axes[1].text(b.get_x()+b.get_width()/2, v+0.01, f'{v:.3f}',
                ha='center', fontsize=8, fontweight='bold')
axes[1].set_ylabel(f'F-score @ {threshold}')
axes[1].set_title('(b) F-score\n(higher = better geometry)', fontsize=10)
axes[1].set_ylim(0, 1.1)
axes[1].tick_params(axis='x', rotation=30)

# (c) PSNR bars
psnr_vals = {}
for name in all_names:
    if name not in all_preds: continue
    mse = ((all_preds[name][m_px] - hr[m_px]) ** 2).mean()
    psnr_vals[name] = 10 * np.log10(1.0 / max(mse, 1e-12))

names_p = list(psnr_vals.keys())
vals_p = [psnr_vals[n] for n in names_p]
colors_p = [all_colors[n] for n in names_p]
bars2 = axes[2].bar(names_p, vals_p, color=colors_p, edgecolor='black', linewidth=0.5)
for b, v in zip(bars2, vals_p):
    axes[2].text(b.get_x()+b.get_width()/2, v+0.2, f'{v:.1f}',
                ha='center', fontsize=8, fontweight='bold')
axes[2].set_ylabel('PSNR (dB)')
axes[2].set_title('(c) PSNR\n(higher = lower pixel error)', fontsize=10)
axes[2].set_ylim(35, max(vals_p) + 3)
axes[2].tick_params(axis='x', rotation=30)

fig.suptitle('§12  Per-method analysis: PSNR similar, but F-score gap = 35pp', fontsize=12, fontweight='bold')
fig.tight_layout()
save_fig(fig, 'fig12_error_analysis')
plt.show()


## §13 Hero Figure — F-score Binary + Surface Shading\nDepth looks the same → geometry differs

In [ ]:
threshold = 1e-3
pass_fail_cmap = ListedColormap(['#d32f2f', '#43a047'])

methods = [('Bicubic', bic_8, C_BIC)]
for name, c in [('EDSR', C_PIX), ('SRResNet', C_PIX2), ('SGNet', C_SGN),
                ('UNet', C_OURS2), ('UNet+DSINE', C_OURS)]:
    if name in preds: methods.append((name, preds[name], c))
methods.append(('GT', hr, '#2E7D32'))
n = len(methods)

cy, cx, cs = 490, 500, 110
crop = (slice(cy-cs, cy+cs), slice(cx-cs, cx+cs))

fig, axes = plt.subplots(4, n, figsize=(n * 2.2, 11), gridspec_kw={'height_ratios': [1,1,1,1]})
for col, (name, pred, clr) in enumerate(methods):
    is_gt = (name == 'GT')
    fw = 'bold' if name in ('UNet', 'UNet+DSINE', 'GT') else 'normal'
    axes[0][col].imshow(to_depth_rgb(pred, vmin, vmax)[crop])
    axes[0][col].set_title(name, fontsize=9, fontweight=fw, color=clr); axes[0][col].axis('off')
    if is_gt:
        axes[1][col].imshow(np.ones((2*cs, 2*cs)), cmap=pass_fail_cmap, vmin=0, vmax=1)
        axes[1][col].text(0.5, 0.5, '100%', transform=axes[1][col].transAxes, ha='center', va='center', fontsize=14, color='white', fontweight='bold')
    else:
        within = (np.abs(pred - hr) < threshold).astype(float)
        axes[1][col].imshow(within[crop], cmap=pass_fail_cmap, vmin=0, vmax=1)
        pct = within[crop][mask[crop] > 0.5].mean() * 100 if mask is not None else within[crop].mean() * 100
        axes[1][col].text(0.5, 0.05, f'{pct:.1f}%', transform=axes[1][col].transAxes, ha='center', va='bottom', fontsize=11, color='white', fontweight='bold', bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.7))
    axes[1][col].axis('off')
    axes[2][col].imshow(shaded_surface(pred[crop]), cmap='gray', vmin=0.3, vmax=1.0); axes[2][col].axis('off')
    if is_gt:
        axes[3][col].imshow(np.ones_like(hr[crop]) * 0.5, cmap='RdBu_r', vmin=-1, vmax=1)
        axes[3][col].text(0.5, 0.5, 'Ref', transform=axes[3][col].transAxes, ha='center', va='center', fontsize=10)
    else:
        axes[3][col].imshow((shaded_surface(pred[crop]) - shaded_surface(hr[crop])) * 5, cmap='RdBu_r', vmin=-0.5, vmax=0.5)
    axes[3][col].axis('off')
for r, lbl in enumerate(['Depth', f'F@{threshold}', 'Surface shading', 'Shading diff x5']):
    axes[r][0].set_ylabel(lbl, fontsize=8, rotation=90, labelpad=8)
    axes[r][0].yaxis.set_visible(True); axes[r][0].tick_params(left=False, labelleft=False)
fig.suptitle(f'§13  Hero: depth looks the same, geometry differs (threshold={threshold})', fontsize=11, fontweight='bold')
fig.tight_layout(h_pad=0.4)
save_fig(fig, 'fig13_hero_fscore_shading')
plt.show()

## §14 Training Convergence

In [ ]:
curve_dirs = {
    'UNet': 'depth_upres_8bit_ch32', 'UNet+DSINE': 'depth_upres_8bit_ch32_normal_w050_dsine',
    'EDSR': 'baseline_edsr', 'SRResNet': 'baseline_srresnet', 'SGNet': 'baseline_sgnet',
}
curve_colors = {'UNet': C_OURS2, 'UNet+DSINE': C_OURS, 'EDSR': C_PIX, 'SRResNet': C_PIX2, 'SGNet': C_SGN}
curve_styles = {'UNet': '-', 'UNet+DSINE': '-', 'EDSR': '--', 'SRResNet': '--', 'SGNet': '-.'}

fig, ax = plt.subplots(figsize=(6, 3.5))
for label, ckdir in curve_dirs.items():
    log_path = os.path.join(CKPT, ckdir, 'train_log.json')
    if os.path.exists(log_path):
        with open(log_path) as f: log = json.load(f)
        epochs = [e['epoch'] for e in log]
        val_l1 = [e.get('val_l1', e.get('val_loss')) for e in log]
        val_l1 = [v for v in val_l1 if v is not None]
        ax.plot(epochs[:len(val_l1)], val_l1, curve_styles[label], color=curve_colors[label], label=label, linewidth=1.3)
        bi = int(np.argmin(val_l1))
        ax.plot(epochs[bi], val_l1[bi], 'o', color=curve_colors[label], markersize=5)
        print(f'  {label}: best val_l1={val_l1[bi]:.5f} @ ep{epochs[bi]}')
ax.set_xlabel('Epoch'); ax.set_ylabel('Val L1'); ax.set_ylim(0, 0.008)
ax.legend(fontsize=7.5, ncol=2); ax.grid(True, alpha=0.3)
fig.suptitle('§14  Training convergence', fontsize=11, fontweight='bold')
fig.tight_layout()
save_fig(fig, 'fig14_convergence')
plt.show()

## §15 Degradation Cross-test (5x5 PSNR)

In [ ]:
import pandas as pd
import matplotlib.patheffects as pe

csv_path = os.path.join(ROOT, 'eval', 'degradation_crosstest_PSNR.csv')
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path, index_col=0)
    mat = df.values.astype(float); labels = list(df.columns)

    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    from matplotlib.colors import LinearSegmentedColormap
    cmap_gray = LinearSegmentedColormap.from_list('gray_blue',
        ['#2c4a6e', '#4a6785', '#7b8fa8', '#adb9ca', '#d6dce4', '#f0f2f5'])

    im = ax.imshow(mat, cmap=cmap_gray, vmin=32, vmax=47, aspect='equal')
    for i_r in range(mat.shape[0]):
        for j_c in range(mat.shape[1]):
            v = mat[i_r, j_c]
            ax.text(j_c, i_r, f'{v:.1f}', ha='center', va='center', fontsize=10,
                    fontweight='bold', color='#111111',
                    path_effects=[pe.withStroke(linewidth=2.5, foreground='white')])

    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels([l.replace('test_','') for l in labels], fontsize=8, rotation=30, ha='right')
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels([l.replace('test_','') for l in labels], fontsize=8)
    ax.set_xlabel('Test degradation'); ax.set_ylabel('Train degradation')

    ax.add_patch(Rectangle((-0.5,-0.5), 3, 3, fill=False, edgecolor='#333333', linestyle='--', linewidth=1.2))
    ax.add_patch(Rectangle((2.5, 2.5), 2, 2, fill=False, edgecolor='#333333', linestyle='--', linewidth=1.2))
    for k in range(mat.shape[0]):
        ax.add_patch(Rectangle((k-0.5, k-0.5), 1, 1, fill=False, edgecolor='#111111', linewidth=1.8))

    cbar = fig.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label('PSNR (dB)', fontsize=9)
    fig.suptitle('Cross-degradation PSNR matrix', fontsize=11, fontweight='bold')
    fig.tight_layout()
    save_fig(fig, 'fig15_degradation_heatmap')
    plt.show()
else:
    print(f'Missing: {csv_path}')


## §16 Zone-Aware Evaluation

In [ ]:
fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(8, 3.5), gridspec_kw={'width_ratios': [1, 1.5]})
depth_vis = to_depth_rgb(hr, vmin, vmax)
overlay = depth_vis.copy()
if mask is not None:
    overlay[mask > 0.5, 0] = np.clip(overlay[mask > 0.5, 0] + 0.2, 0, 1)
ax_img.imshow(overlay); ax_img.set_title('Zone mask (|yaw| <= 20 deg)'); ax_img.axis('off')

methods_bar = ['Bicubic', 'EDSR', 'SRResNet', 'SGNet', 'UNet', 'UNet+DSINE']
za_psnr = [42.5, 46.8, 47.3, 48.9, 47.0, 47.2]
colors_bar = [C_BIC, C_PIX, C_PIX2, C_SGN, C_OURS2, C_OURS]
bars = ax_bar.bar(methods_bar, za_psnr, color=colors_bar, edgecolor='black', linewidth=0.5)
for b, v in zip(bars, za_psnr):
    ax_bar.text(b.get_x()+b.get_width()/2, v+0.1, f'{v:.1f}', ha='center', fontsize=7, fontweight='bold')
ax_bar.set_ylabel('Zone-aware PSNR (dB)'); ax_bar.set_ylim(40, 51)
ax_bar.tick_params(axis='x', rotation=30); ax_bar.set_title('Zone-aware PSNR')
fig.suptitle('§16  Zone-aware evaluation', fontsize=11, fontweight='bold')
fig.tight_layout(); save_fig(fig, 'fig16_zone_aware'); plt.show()

## §17 Mesh Quality: F-score + Hausdorff

In [ ]:
mq_path = os.path.join(ROOT, 'eval', 'mesh_quality.csv')
if os.path.exists(mq_path):
    mq = pd.read_csv(mq_path)
    name_map = {
        'LR-bicubic-8bit': 'Bicubic', 'EDSR-light': 'EDSR', 'SRResNet': 'SRResNet',
        'SGNet': 'SGNet', 'UNet-8bit-ch32': 'UNet',
        'UNet-8bit_ch32_normal_w050_dsine': 'UNet+DSINE'
    }
    mq['short'] = mq['method'].map(name_map)
    mq = mq.dropna(subset=['short'])
    order = ['Bicubic', 'EDSR', 'SRResNet', 'SGNet', 'UNet', 'UNet+DSINE']
    mq['sort'] = mq['short'].map({n: i for i, n in enumerate(order)})
    mq = mq.sort_values('sort')
    colors = [C_BIC, C_PIX, C_PIX2, C_SGN, C_OURS2, C_OURS]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
    b1 = ax1.bar(mq['short'], mq['f_score_001'], color=colors[:len(mq)], edgecolor='black', linewidth=0.5)
    for b, v in zip(b1, mq['f_score_001']):
        ax1.text(b.get_x()+b.get_width()/2, v+0.005, f'{v:.3f}', ha='center', fontsize=7, fontweight='bold')
    ax1.set_title('F-score @ 1e-3 (higher=better)'); ax1.set_ylim(0.55, 1.05)
    ax1.tick_params(axis='x', rotation=30)

    b2 = ax2.bar(mq['short'], mq['hausdorff'], color=colors[:len(mq)], edgecolor='black', linewidth=0.5)
    for b, v in zip(b2, mq['hausdorff']):
        ax2.text(b.get_x()+b.get_width()/2, v+0.001, f'{v:.4f}', ha='center', fontsize=7, fontweight='bold')
    ax2.set_title('Hausdorff (lower=better)')
    ax2.tick_params(axis='x', rotation=30)

    fig.suptitle('§17  Mesh quality — F-score gap', fontsize=11, fontweight='bold')
    fig.tight_layout()
    save_fig(fig, 'fig17_mesh_quality')
    plt.show()
else:
    print(f'Missing: {mq_path}')

## §18 PSNR vs F-score Scatter (Teaser)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
scatter_pts = {
    'Bicubic':    {'psnr': 42.45, 'f': 0.998, 'c': C_BIC,  'mk': 's', 'ms': 80},
    'EDSR':       {'psnr': 46.78, 'f': 0.637, 'c': C_PIX,  'mk': 'D', 'ms': 70},
    'SRResNet':   {'psnr': 47.25, 'f': 0.647, 'c': C_PIX2, 'mk': 'o', 'ms': 70},
    'SGNet':      {'psnr': 48.89, 'f': 0.654, 'c': C_SGN,  'mk': '^', 'ms': 80},
    'UNet':       {'psnr': 46.97, 'f': 0.999, 'c': C_OURS2,'mk': 'D', 'ms': 90},
    'UNet+DSINE': {'psnr': 47.19, 'f': 0.999, 'c': C_OURS, 'mk': 's', 'ms': 100},
}
for name, p in scatter_pts.items():
    ax.scatter(p['psnr'], p['f'], c=p['c'], marker=p['mk'], s=p['ms'], edgecolors='black', linewidth=0.5, zorder=5)
    offsets = {'Bicubic': (-0.5, 0.01), 'SGNet': (0.3, -0.03), 'UNet': (0.3, -0.02),
               'UNet+DSINE': (-2.5, 0.01), 'EDSR': (-0.5, -0.025), 'SRResNet': (0.3, 0.01)}
    dx, dy = offsets.get(name, (0.3, 0))
    ax.annotate(name, (p['psnr'], p['f']), xytext=(p['psnr']+dx, p['f']+dy), fontsize=7.5, color=p['c'], fontweight='bold')
ax.annotate('35 pp gap', xy=(47.1, 0.82), fontsize=9, color='#c62828', fontweight='bold', ha='center')
ax.annotate('', xy=(47.1, 0.999), xytext=(47.1, 0.654), arrowprops=dict(arrowstyle='->', color='#c62828', lw=1.5))
ax.set_xlabel('Zone-aware PSNR (dB)'); ax.set_ylabel('F-score @ $10^{-3}$')
ax.set_xlim(41.5, 50); ax.set_ylim(0.55, 1.02); ax.grid(True, alpha=0.3)
fig.suptitle('§18  PSNR vs F-score: PixelShuffle wins PSNR, loses geometry', fontsize=10, fontweight='bold')
fig.tight_layout(); save_fig(fig, 'fig18_psnr_vs_fscore'); plt.show()

## §19 Parameters vs Quality

In [ ]:
pts = [
    ('SwinIR', 0.23, 11.95, C_EXT, 'v', 70), ('DORNet', 3.05, 42.14, C_EXT, '^', 70),
    ('EDSR', 1.52, 46.78, C_PIX, 'o', 70), ('SRResNet', 1.53, 47.25, C_PIX2, 'o', 70),
    ('SGNet', 5.4, 48.89, C_SGN, 's', 80),
    ('UNet', 7.77, 46.97, C_OURS2, 'D', 90), ('UNet+DSINE', 7.78, 47.19, C_OURS, 's', 100),
]
fig, ax = plt.subplots(figsize=(5.5, 3.5))
for name, params, psnr, c, mk, ms in pts:
    ax.scatter(params, psnr, c=c, marker=mk, s=ms, edgecolors='black', linewidth=0.5, zorder=5)
    ax.annotate(name, (params, psnr), xytext=(5, 5), textcoords='offset points', fontsize=7.5, color=c, fontweight='bold')
ax.set_xscale('log'); ax.set_xlabel('Parameters (M)'); ax.set_ylabel('PSNR (dB)')
ax.set_ylim(10, 52); ax.grid(True, alpha=0.3)
fig.suptitle('§19  Model size vs PSNR', fontsize=11, fontweight='bold')
fig.tight_layout(); save_fig(fig, 'fig19_params_vs_quality'); plt.show()

## §20 Summary

In [ ]:
flist = sorted([f for f in os.listdir(OUT) if f.endswith('.pdf')])
print(f'Total: {len(flist)} PDF figures in {OUT}')
for f in flist:
    size_kb = os.path.getsize(os.path.join(OUT, f)) / 1024
    print(f'  {f:45s}  {size_kb:6.1f} KB')